In [23]:
!pip install "mrjob<0.8"
from mrjob.job import MRJob
with open("input.txt", "w") as f:
    f.write("hadoop is fast\nhadoop is scalable")

Q 1

In [24]:
# Input
data = ["hadoop is fast", "hadoop is scalable"]

# Mapper
mapped = []
for line in data:
    for word in line.split():
        mapped.append((word, 1))

# Shuffle
shuffle = {}
for word, count in mapped:
    shuffle.setdefault(word, []).append(count)

# Reducer
result = {}
for word, counts in shuffle.items():
    result[word] = sum(counts)

print(result)


{'hadoop': 2, 'is': 2, 'fast': 1, 'scalable': 1}


In [25]:
%%writefile wordcount.py
from mrjob.job import MRJob

class WordCount(MRJob):
    def mapper(self, _, line):
        for word in line.split():
            yield word, 1

    def reducer(self, key, values):
        yield key, sum(values)

if __name__ == "__main__":
    WordCount.run()


Overwriting wordcount.py


In [26]:
!python wordcount.py input.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/wordcount.root.20260419.173312.819548
Running step 1 of 1...
job output is in /tmp/wordcount.root.20260419.173312.819548/output
Streaming final output from /tmp/wordcount.root.20260419.173312.819548/output...
"is"	2
"scalable"	1
"fast"	1
"hadoop"	2
Removing temp directory /tmp/wordcount.root.20260419.173312.819548...


Q2

In [27]:
data = "big data"

mapped = [(c,1) for c in data if c != " "]

shuffle = {}
for k,v in mapped:
    shuffle.setdefault(k, []).append(v)

print({k: sum(v) for k,v in shuffle.items()})

{'b': 1, 'i': 1, 'g': 1, 'd': 1, 'a': 2, 't': 1}


In [29]:
%%writefile charcount.py
from mrjob.job import MRJob

class CharCount(MRJob):
    def mapper(self, _, line):
        for c in line.replace(" ", ""):
            yield c, 1

    def reducer(self, key, values):
        yield key, sum(values)

if __name__ == "__main__":
    CharCount.run()

Writing charcount.py


In [30]:
!python charcount.py input.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/charcount.root.20260419.173756.950879
Running step 1 of 1...
job output is in /tmp/charcount.root.20260419.173756.950879/output
Streaming final output from /tmp/charcount.root.20260419.173756.950879/output...
"d"	2
"e"	1
"f"	1
"h"	2
"i"	2
"l"	2
"o"	4
"p"	2
"s"	4
"t"	1
"a"	5
"b"	1
"c"	1
Removing temp directory /tmp/charcount.root.20260419.173756.950879...


Q3

In [31]:
data = "data science data big data".split()

mapped = [(w,(len(w),1)) for w in data]

shuffle = {}
for k,v in mapped:
    shuffle.setdefault(k, []).append(v)

result = {}
for k,vals in shuffle.items():
    total = sum(v[0] for v in vals)
    count = sum(v[1] for v in vals)
    result[k] = total/count

print(result)

{'data': 4.0, 'science': 7.0, 'big': 3.0}


In [44]:
%%writefile top5.py
from mrjob.job import MRJob
from collections import Counter

class Top5(MRJob):

    def mapper(self, _, line):
        for word in line.split():
            yield word, 1

    def reducer(self, key, values):
        yield None, (key, sum(values))

    def reducer_final(self):
        counter = Counter()
        for key, value in self.reducer_init_data:
            word, count = value
            counter[word] = count

        for word, count in counter.most_common(5):
            yield word, count

if __name__ == "__main__":
    Top5.run()

Overwriting top5.py


In [35]:
!python avgword.py input.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/avgword.root.20260419.174057.381787
Running step 1 of 1...
job output is in /tmp/avgword.root.20260419.174057.381787/output
Streaming final output from /tmp/avgword.root.20260419.174057.381787/output...
"is"	2.0
"scalable"	8.0
"fast"	4.0
"hadoop"	6.0
Removing temp directory /tmp/avgword.root.20260419.174057.381787...


Q4


In [36]:
data = "hadoop mapreduce spark".split()

total = sum(len(w) for w in data)
count = len(data)

print(total/count)

6.666666666666667


In [37]:
%%writefile globalavg.py
from mrjob.job import MRJob

class GlobalAvg(MRJob):
    def mapper(self, _, line):
        for w in line.split():
            yield "avg", (len(w),1)

    def reducer(self, key, values):
        total,count = 0,0
        for l,c in values:
            total += l
            count += c
        yield key, total/count

if __name__ == "__main__":
    GlobalAvg.run()

Writing globalavg.py


In [38]:
!python globalavg.py input.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/globalavg.root.20260419.174921.174323
Running step 1 of 1...
job output is in /tmp/globalavg.root.20260419.174921.174323/output
Streaming final output from /tmp/globalavg.root.20260419.174921.174323/output...
"avg"	4.666666666666667
Removing temp directory /tmp/globalavg.root.20260419.174921.174323...


Q5

In [39]:
from google.colab import files
uploaded = files.upload()

Saving shakespeare.txt to shakespeare.txt


In [58]:
import re
from collections import Counter

with open("shakespeare.txt") as f:
    text = f.read().lower()

# Remove punctuation
words = re.findall(r'\b[a-z]+\b', text)

counter = Counter(words)


print(counter.most_common(5))

[('the', 27843), ('and', 26847), ('i', 22538), ('to', 19882), ('of', 18307)]


In [64]:
%%writefile top5.py
from mrjob.job import MRJob
from mrjob.step import MRStep
from collections import Counter
import re

class Top5(MRJob):

    def mapper(self, _, line):
        # clean words (important for marks)
        for word in re.findall(r'\b[a-z]+\b', line.lower()):
            yield word, 1

    def reducer(self, key, values):
        yield None, (key, sum(values))

    # STEP 2
    def reducer_top5_init(self):
        self.counter = Counter()

    def reducer_top5(self, _, word_counts):
        for word, count in word_counts:
            self.counter[word] += count

    def reducer_top5_final(self):
        for word, count in self.counter.most_common(5):
            yield word, count

    def steps(self):
        return [
            MRStep(mapper=self.mapper,
                   reducer=self.reducer),
            MRStep(reducer_init=self.reducer_top5_init,
                   reducer=self.reducer_top5,
                   reducer_final=self.reducer_top5_final)
        ]

if __name__ == "__main__":
    Top5.run()

Overwriting top5.py


In [65]:
!python top5.py shakespeare.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/top5.root.20260419.181130.172951
Running step 1 of 2...
Running step 2 of 2...
job output is in /tmp/top5.root.20260419.181130.172951/output
Streaming final output from /tmp/top5.root.20260419.181130.172951/output...
"the"	27843
"and"	26847
"i"	22538
"to"	19882
"of"	18307
Removing temp directory /tmp/top5.root.20260419.181130.172951...


Q6

In [66]:
data = ["A 80","B 70","A 90","B 60","A 100"]

mapped = [(x.split()[0], (int(x.split()[1]),1)) for x in data]

shuffle = {}
for k,v in mapped:
    shuffle.setdefault(k, []).append(v)

print({k: sum(v[0] for v in vals)/sum(v[1] for v in vals) for k,vals in shuffle.items()})

{'A': 90.0, 'B': 65.0}


In [67]:
%%writefile avg_marks.py
from mrjob.job import MRJob

class AvgMarks(MRJob):

    def mapper(self, _, line):
        student, marks = line.split()
        yield student, (int(marks), 1)

    def reducer(self, student, values):
        total = 0
        count = 0
        for marks, c in values:
            total += marks
            count += c
        yield student, total / count

if __name__ == "__main__":
    AvgMarks.run()

Writing avg_marks.py


In [69]:
with open("marks.txt", "w") as f:
    f.write("""A 80
B 70
A 90
B 60
A 100""")

In [70]:
!python avg_marks.py marks.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/avg_marks.root.20260419.181429.456025
Running step 1 of 1...
job output is in /tmp/avg_marks.root.20260419.181429.456025/output
Streaming final output from /tmp/avg_marks.root.20260419.181429.456025/output...
"B"	65.0
"A"	90.0
Removing temp directory /tmp/avg_marks.root.20260419.181429.456025...


Q7

In [71]:
data = ["HR 50000","IT 70000","HR 60000","IT 80000"]

mapped = [(x.split()[0], (int(x.split()[1]),1)) for x in data]

shuffle = {}
for k,v in mapped:
    shuffle.setdefault(k, []).append(v)

avg = {k: sum(v[0] for v in vals)/len(vals) for k,vals in shuffle.items()}

print(avg)
print("Highest:", max(avg, key=avg.get))

{'HR': 55000.0, 'IT': 75000.0}
Highest: IT


In [81]:
%%writefile dept_salary.py
from mrjob.job import MRJob
from mrjob.step import MRStep

class DeptSalary(MRJob):

    def mapper(self, _, line):
        dept, salary = line.split()
        yield dept, (int(salary), 1)

    def reducer_avg(self, dept, values):
        total = 0
        count = 0
        for sal, c in values:
            total += sal
            count += c

        avg = total / count

        # emit avg
        yield dept, avg

        # send for max calculation
        yield "MAX", (dept, avg)

    def reducer_max(self, key, values):
        if key == "MAX":
            max_dept = None
            max_avg = 0
            for dept, avg in values:
                if avg > max_avg:
                    max_avg = avg
                    max_dept = dept
            yield "Highest Paid Dept", (max_dept, max_avg)
        else:
            # pass avg values forward
            for v in values:
                yield key, v

    def steps(self):
        return [
            MRStep(mapper=self.mapper,
                   reducer=self.reducer_avg),
            MRStep(reducer=self.reducer_max)
        ]

if __name__ == "__main__":
    DeptSalary.run()

Overwriting dept_salary.py


In [82]:
with open("salary.txt", "w") as f:
    f.write("""HR 50000
IT 70000
HR 60000
IT 80000""")

In [83]:
!python dept_salary.py salary.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/dept_salary.root.20260419.182225.125681
Running step 1 of 2...
Running step 2 of 2...
job output is in /tmp/dept_salary.root.20260419.182225.125681/output
Streaming final output from /tmp/dept_salary.root.20260419.182225.125681/output...
"Highest Paid Dept"	["IT", 75000.0]
"HR"	55000.0
"IT"	75000.0
Removing temp directory /tmp/dept_salary.root.20260419.182225.125681...


Q8

In [84]:
data = ["New York,38","London,29","Tokyo,35","New York,32","Delhi,45","Ambala,35"]

mapped = [(x.split(",")[0], (int(x.split(",")[1]),1)) for x in data]

shuffle = {}
for k,v in mapped:
    shuffle.setdefault(k, []).append(v)

print({k: sum(v[0] for v in vals)/len(vals) for k,vals in shuffle.items()})

{'New York': 35.0, 'London': 29.0, 'Tokyo': 35.0, 'Delhi': 45.0, 'Ambala': 35.0}


In [85]:
%%writefile temp_avg.py
from mrjob.job import MRJob

class TempAvg(MRJob):

    def mapper(self, _, line):
        city, temp = line.split(",")
        yield city, (int(temp), 1)

    def reducer(self, city, values):
        total = 0
        count = 0
        for temp, c in values:
            total += temp
            count += c
        yield city, total / count

if __name__ == "__main__":
    TempAvg.run()

Writing temp_avg.py


In [87]:
with open("temp.txt", "w") as f:
    f.write("""New York,38
London,29
Tokyo,35
New York,32
Yamuna Nagar,45
Saharanpur,35""")

In [88]:
!python temp_avg.py temp.txt

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/temp_avg.root.20260419.182727.001833
Running step 1 of 1...
job output is in /tmp/temp_avg.root.20260419.182727.001833/output
Streaming final output from /tmp/temp_avg.root.20260419.182727.001833/output...
"Saharanpur"	35.0
"Tokyo"	35.0
"Yamuna Nagar"	45.0
"London"	29.0
"New York"	35.0
Removing temp directory /tmp/temp_avg.root.20260419.182727.001833...


Q9

In [93]:
import pandas as pd

df = pd.read_csv("file.csv")

df.groupby("Country")["AverageTemperature"].mean().sort_values(ascending=False)
df.head()

,dt,AverageTemperature,AverageTemperatureUncertainty,City,Country,Latitude,Longitude
0,1849-01-01,26.704,1.435,Abidjan,Côte D'Ivoire,5.63N,3.23W
1,1849-02-01,27.434,1.362,Abidjan,Côte D'Ivoire,5.63N,3.23W
2,1849-03-01,28.101,1.612,Abidjan,Côte D'Ivoire,5.63N,3.23W
3,1849-04-01,26.140,1.387,Abidjan,Côte D'Ivoire,5.63N,3.23W
4,1849-05-01,25.427,1.200,Abidjan,Côte D'Ivoire,5.63N,3.23W


In [96]:
df = df.dropna(subset=["AverageTemperature"])
result = df.groupby("Country")["AverageTemperature"].mean()
print(result)

Country
Afghanistan                           14.342919
Angola                                23.693046
Australia                             15.190055
Bangladesh                            25.490568
Brazil                                22.847555
Burma                                 26.735193
Canada                                 5.109462
Chile                                  5.692277
China                                 11.793666
Colombia                              20.892248
Congo (Democratic Republic Of The)    23.866441
Côte D'Ivoire                         26.163737
Dominican Republic                    25.976777
Egypt                                 20.900406
Ethiopia                              17.525073
France                                10.402644
Germany                                8.916234
India                                 25.809309
Indonesia                             26.659057
Iran                                  12.571992
Iraq                            

In [107]:
%%writefile temp_country.py
from mrjob.job import MRJob
import csv

class TempCountry(MRJob):

    def mapper(self, _, line):
        try:
            row = next(csv.reader([line]))

            if row[0] == "dt":
                return


            if len(row) > 4 and row[1] and row[4]:
                country = row[4]
                temp = float(row[1])
                yield country, (temp, 1)

        except:
            pass

    def reducer(self, country, values):
        total = 0
        count = 0

        for temp, c in values:
            total += temp
            count += c

        if count > 0:
            yield country, total / count

if __name__ == "__main__":
    TempCountry.run()

Overwriting temp_country.py


In [108]:
!python temp_country.py file.csv

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/temp_country.root.20260419.184637.853537
Running step 1 of 1...
job output is in /tmp/temp_country.root.20260419.184637.853537/output
Streaming final output from /tmp/temp_country.root.20260419.184637.853537/output...
"Colombia"	20.892248063952056
"Congo (Democratic Republic Of The)"	23.866440922190183
"Dominican Republic"	25.976776619845957
"Egypt"	20.900406225165536
"Ethiopia"	17.525072662298964
"France"	10.402644346178146
"Germany"	8.916233733417545
"India"	25.809309238455356
"Indonesia"	26.65905694518359
"Iran"	12.571992111368914
"Iraq"	22.61434568965519
"Italy"	11.965501895135807
"Japan"	13.658246913580218
"Kenya"	16.081395113230037
"Mexico"	15.717422377622395
"Morocco"	17.184157858613602
"Nigeria"	26.361322692307702
"Pakistan"	24.811197226502376
"Peru"	16.769119658119664
"Philippines"	26.448334487877297
"Russia"	3.9588776058117565
"Saudi Arabia"	26.427309387966

# New Section